# FRB Classifier FPGA Synthesis - Pooled Model

This notebook synthesizes the pooled FRB classifier for FPGA deployment using hls4ml.

## Step 1: Imports

In [ ]:
import hls4ml
import numpy as np
import os

# Use tensorflow.keras for loading the model
import tensorflow as tf
from tensorflow import keras

print(f"TensorFlow version: {tf.__version__}")
print(f"hls4ml version: {hls4ml.__version__}")

## Step 2: Rebuild Model Architecture and Load Weights

Had difficulties loading the model due to Keras differences *(Keras 3 on home machine/Keras 2 on EC2)*. <br>
Solution: Create the CNN in notebook and load the weights via the frb_classifer_poooled.h5 file.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Rebuild the EXACT same architecture as your training notebook
def create_simple_cnn(input_shape):
    model = Sequential()
    model.add(Conv2D(2, (3, 3), activation='relu', input_shape=input_shape))
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.5))
    model.add(Conv2D(2, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Flatten())
    model.add(Dense(2, activation='softmax'))
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Create the model
input_shape = (64, 64, 1)
model = create_simple_cnn(input_shape)

# Load the trained weights
model.load_weights('frb_classifier_pooled.h5')

print("Model architecture rebuilt")
print("Weights loaded successfully :)")
print()
model.summary()

## Step 3: Configure hls4ml

Key settings:
- **Precision:** `ap_fixed<16,6>` - 16 bits total, 6 integer bits
- **ReuseFactor:** 1 for lowest latency
- **Strategy:** Latency (maximize parallelism)
- **IOType:** io_stream (required for CNNs)

In [ ]:
# Create hls4ml configuration
config = hls4ml.utils.config_from_keras_model(model, granularity='name')

# Global settings
config['Model']['Precision'] = 'ap_fixed<16,6>'
config['Model']['ReuseFactor'] = 1
config['Model']['Strategy'] = 'Latency'

# Print configuration
print("hls4ml Configuration:")
print("="*60)
print(f"Precision: {config['Model']['Precision']}")
print(f"ReuseFactor: {config['Model']['ReuseFactor']}")
print(f"Strategy: {config['Model']['Strategy']}")
print()
print("Layer configurations:")
for layer_name, layer_cfg in config['LayerName'].items():
    print(f"  {layer_name}: {layer_cfg}")

## Step 4: Convert to HLS Model

In [ ]:
# Output directory
output_dir = 'frb_pooled_hls'

# Convert the model
print("Converting Keras model to HLS...")

hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=config,
    output_dir=output_dir,
    project_name='frb_classifier',
    part='xcvu9p-flga2104-2L-e',
    clock_period=5,
    io_type='io_stream',  # Required for CNN
    backend='Vitis'
)

print()
print(f"HLS model created")
print(f"Output directory: {output_dir}")

## Step 5: Compile and Test (C++ Emulation)

In [ ]:
# Compile for software emulation
print("Compiling...")
hls_model.compile()
print("Compilation complete!")

## Step 6: Write HLS Project Files

In [ ]:
# Write the HLS project
print("Writing HLS project files...")
hls_model.write()
print(f"Project written to: {output_dir}/")

## Step 7: Run Synthesis

This is the actual FPGA synthesis. <br><br>
The standard hls4ml workflow uses **hls_model.build()** to run synthesis, but this fails on newer Vitis HLS versions (2024+). The **build()** command internally calls ```vivado_hls```, which was the old tool name. Xilinx renamed it to ```vitis-run --mode ``` in recent versions, but hls4ml hasn't updated to reflect this change. Additionally, hls4ml tries to source an outdated part. Rather than modifying the hls4ml source code or messing around with upgrading and downgrading versions + directories, we run synthesis manually using a TCL script.

In [ ]:
import subprocess
import time

print("="*70)
print("STARTING FPGA SYNTHESIS")
print("="*70)
print(f"Start time: {time.strftime('%H:%M:%S')}")
print()

# Change to output directory
original_dir = os.getcwd()
os.chdir(output_dir)

# Create synthesis TCL script
synth_tcl = '''
open_project frb_classifier_prj # Opens/Creates HLS project directoru
set_top frb_classifier # Specifies the top-level function to synthesize

add_files firmware/frb_classifier.cpp -cflags "-std=c++0x" # Adds C++ code generated by hls4ml
add_files -tb frb_classifier_test.cpp -cflags "-std=c++0x" # Adds testbench files for verification
add_files -tb firmware/weights # Adds weights
add_files -tb tb_data # Adds tb_data

open_solution "solution1" # Creates a solution
set_part {xcvu9p-flga2104-2L-e} # Targets the Xilinx VU9P FPGA (this was causing trouble in synthesis before)
create_clock -period 5 -name default # Sets 5ns clock period

config_compile -pipeline_loops 64 # Automatically pipelines loops w/ >64 iterations for better thruput

puts "Starting C synthesis..."
csynth_design # Runs the actual C-to-RTL synthesis

puts "Synthesis complete!"
exit
'''

with open('run_synthesis.tcl', 'w') as f:
    f.write(synth_tcl)

# Create bash script
bash_script = '''#!/bin/bash
# Source Xilinx Vitis HLS environment
# (tries two common install paths for compatibility across different setups)
source /opt/Xilinx/Vitis/2025.1/settings64.sh 2>/dev/null || source /opt/Xilinx/2025.1/Vitis/settings64.sh 2>/dev/null || true
# Run synthesis using vitis-run (Vitis 2024+ replacement for vivado_hls)
# --mode hls: run in HLS mode
# --tcl: execute our TCL script
# tee: save output to log file while also printing to screen
vitis-run --mode hls --tcl run_synthesis.tcl 2>&1 | tee synthesis.log
'''

with open('run_synthesis.sh', 'w') as f:
    f.write(bash_script)
os.chmod('run_synthesis.sh', 0o755)

print("Synthesis scripts created")
print("Running synthesis...")
print()

start_time = time.time()

# Run synthesis
result = subprocess.run(['./run_synthesis.sh'], shell=True, capture_output=False)

elapsed = (time.time() - start_time) / 60

print()
print("="*70)
print(f"SYNTHESIS FINISHED")
print(f"Total time: {elapsed:.1f} minutes")
print("="*70)

# Return to original directory
os.chdir(original_dir)